# Tokamak X-Point with Parametric TikZ Plots

This tutorial builds on [Mathematical
Expressions](https://max-models.github.io/tikzfigure/tutorials/tutorial-10-math-expressions/) and focuses on the plain TikZ
plotting API by building up to a stylized **tokamak X-point / separatrix**
drawing in a poloidal cross-section.

The goal is not to derive a full equilibrium solver inside TikZ. Instead, the
point is to show how declared PGF functions, parametric expressions, and looped
sampling can be combined to produce the kind of contour structure you see in an
X-point divertor configuration.

Along the way we will use:

- `fig.plot(...)` for a single sampled plot
- `loop.plot(...)` for families of curves
- callable objects returned by `fig.declare_function(...)`
- `func(name, ...)` when you still want the string-based interface

These plots are regular TikZ `\draw ... plot(...)` commands, not pgfplots axis
plots.

In [ ]:
from tikzfigure import TikzFigure, colors
from tikzfigure.math import Var, cos, exp, ln, max, sin, sqrt

## A first sampled curve

The simplest case is a regular function graph. Here `fig.plot(...)` samples a
single expression over a domain. This is the basic building block behind all of
the more complicated flux-surface-like contours later in the tutorial.

In [ ]:
fig = TikzFigure(figsize=(8, 4))

x = Var("x")

fig.draw([(-2.4, 0), (2.4, 0)], color="gray")
fig.draw([(0, -0.8), (0, 0.8)], color="gray")
fig.plot(
    sin(90 * x) / 2,
    variable="x",
    domain=(-2, 2),
    samples=120,
    smooth=True,
    options=[colors.blue],
)

fig.show()

In [ ]:
print(fig)

In [ ]:
print(fig.generate_standalone())

When only one expression is passed, TikZ uses the sampling variable as the
x-coordinate and the expression as the y-coordinate.

## Declaring a custom shaping function

You can declare helper functions once and reuse them in later plots. For this
tutorial we will use a hyperbolic-tangent-like shaping function that will later
help us bend contours into an X-point geometry.

In [ ]:
fig = TikzFigure(figsize=(8, 4))

th = fig.declare_function(
    "th",
    "x",
    (1 - exp(-2 * Var("x"))) / (1 + exp(-2 * Var("x"))),
)

x = Var("x")

fig.draw([(-3.4, 0), (3.4, 0)], color="gray")
fig.draw([(0, -1.1), (0, 1.1)], color="gray")
fig.plot(
    th(x),
    variable="x",
    domain=(-3, 3),
    samples=150,
    smooth=True,
    options=[colors.Azure4],
    line_width="0.8pt",
)

fig.show()

In [ ]:
print(fig)

In [ ]:
print(fig.generate_standalone())

The object returned by `fig.declare_function(...)` is callable, so the most
natural way to use it is `th(x)`. The older `func("th", x)` style still works
when you want it.

## Your first closed flux-surface-like curve

If you pass both `x` and `y` expressions, the plot becomes parametric. This is
the key step from ordinary function graphs to closed magnetic-surface-like
contours.

In [ ]:
fig = TikzFigure(figsize=(7, 7))

fig.variable("scale", 1.25)
fig.variable("radius", 1.6)
arcth = fig.declare_function(
    "arcth",
    "x",
    0.5 * ln((1 + Var("x")) / (1 - Var("x"))),
)
th = fig.declare_function(
    "th",
    "x",
    (1 - exp(-2 * Var("x"))) / (1 + exp(-2 * Var("x"))),
)

theta = Var("theta")
radius = Var("radius")

curve_x = Var("scale") * arcth(th(radius) * cos(theta))
curve_y = Var("scale") * arcth(th(radius) * sin(theta))

fig.plot(
    curve_x,
    curve_y,
    variable="theta",
    domain=(0, 360),
    samples=140,
    smooth=True,
    options=[colors.Azure4],
)

fig.show()

In [ ]:
print(fig)

In [ ]:
print(fig.generate_standalone())

Now the sampling variable is `\theta`, and both coordinates are computed from
it. You can think of `\theta` here as a poloidal-angle-like parameter tracing a
single closed contour.

## Building a family of nested flux surfaces

The same parametric expression becomes much more interesting when combined with
`fig.loop(...)`.

In [ ]:
fig = TikzFigure(figsize=(8, 8))

fig.variable("scale", 1.25)
arcth = fig.declare_function(
    "arcth",
    "x",
    0.5 * ln((1 + Var("x")) / (1 - Var("x"))),
)
th = fig.declare_function(
    "th",
    "x",
    (1 - exp(-2 * Var("x"))) / (1 + exp(-2 * Var("x"))),
)

theta = Var("theta")

with fig.loop("i", range(21)) as i:
    radius = 0.2 * i
    curve_x = Var("scale") * arcth(th(radius) * cos(theta))
    curve_y = Var("scale") * arcth(th(radius) * sin(theta))
    fig.plot(
        curve_x,
        curve_y,
        variable="theta",
        domain=(0, 360),
        samples=100,
        smooth=True,
        options=[colors.Azure4],
    )

fig.show()

In [ ]:
print(fig)

In [ ]:
print(fig.generate_standalone())

Each loop iteration gives TikZ a different integer value of `\i`. Because the
context variable `i` now behaves like that TikZ variable directly, you can
derive the physical radius inside the block with `radius = 0.2 * i` and then use
the ordinary `fig.plot(...)` API inside the loop body. That keeps the loop
compact while still producing a whole family of nested contours. In the tokamak
picture, these read naturally as flux surfaces moving outward from the core.

## Adding the crossing family near the X-point

Next, add the second family by keeping `\theta` fixed and varying `\r`. Once
both families are drawn, the contour structure starts to look like the crossing
geometry around an X-point.

In [ ]:
fig = TikzFigure(figsize=(8, 8))

fig.variable("scale", 1.25)
arcth = fig.declare_function(
    "arcth",
    "x",
    0.5 * ln((1 + Var("x")) / (1 - Var("x"))),
)
th = fig.declare_function(
    "th",
    "x",
    (1 - exp(-2 * Var("x"))) / (1 + exp(-2 * Var("x"))),
)

theta = Var("theta")

with fig.loop("i", range(21)) as i:
    radius = 0.2 * i
    curve_x_loop = Var("scale") * arcth(th(radius) * cos(theta))
    curve_y_loop = Var("scale") * arcth(th(radius) * sin(theta))
    fig.plot(
        curve_x_loop,
        curve_y_loop,
        variable="theta",
        domain=(0, 360),
        samples=100,
        smooth=True,
        options=[colors.Azure4],
    )

with fig.loop("theta", range(0, 361, 9)) as theta:
    r = Var("r")
    curve_x_cross = Var("scale") * arcth(th(r) * cos(theta))
    curve_y_cross = Var("scale") * arcth(th(r) * sin(theta))
    fig.plot(
        curve_x_cross,
        curve_y_cross,
        variable="r",
        domain=(0, 4),
        samples=15,
        smooth=True,
        options=[colors.Azure4],
    )

fig.show()

In [ ]:
print(fig)

In [ ]:
print(fig.generate_standalone())

## The full X-point / separatrix sketch

Now add a second pass with thicker red guide curves to highlight a few selected
surfaces and the separatrix-like branches.

In [ ]:
fig = TikzFigure(figsize=(9, 9))

fig.variable("scale", 1.25)
arcth = fig.declare_function(
    "arcth",
    "x",
    0.5 * ln((1 + Var("x")) / (1 - Var("x"))),
)
th = fig.declare_function(
    "th",
    "x",
    (1 - exp(-2 * Var("x"))) / (1 + exp(-2 * Var("x"))),
)

theta = Var("theta")

fine_radii = range(21)
major_radii = range(5, 21, 5)
fine_angles = range(0, 361, 9)
major_angles = range(0, 360, 30)

with fig.loop("i", fine_radii) as i:
    radius = 0.2 * i
    curve_x_loop = Var("scale") * arcth(th(radius) * cos(theta))
    curve_y_loop = Var("scale") * arcth(th(radius) * sin(theta))
    fig.plot(
        curve_x_loop,
        curve_y_loop,
        variable="theta",
        domain=(0, 360),
        samples=100,
        smooth=True,
        options=[colors.Azure4],
    )

with fig.loop("theta", fine_angles) as theta:
    r = Var("r")
    curve_x_cross = Var("scale") * arcth(th(r) * cos(theta))
    curve_y_cross = Var("scale") * arcth(th(r) * sin(theta))
    fig.plot(
        curve_x_cross,
        curve_y_cross,
        variable="r",
        domain=(0, 4),
        samples=15,
        smooth=True,
        options=[colors.Azure4],
    )

with fig.loop("i", major_radii) as i:
    radius = 0.2 * i
    curve_x_loop = Var("scale") * arcth(th(radius) * cos(theta))
    curve_y_loop = Var("scale") * arcth(th(radius) * sin(theta))
    fig.plot(
        curve_x_loop,
        curve_y_loop,
        variable="theta",
        domain=(0, 360),
        samples=120,
        smooth=True,
        options=[colors.Red3],
        line_width="0.9pt",
    )

with fig.loop("theta", major_angles) as theta:
    r = Var("r")
    curve_x_cross = Var("scale") * arcth(th(r) * cos(theta))
    curve_y_cross = Var("scale") * arcth(th(r) * sin(theta))
    fig.plot(
        curve_x_cross,
        curve_y_cross,
        variable="r",
        domain=(0, 4),
        samples=25,
        smooth=True,
        options=[colors.Red3],
        line_width="0.9pt",
    )

fig.show()

In [ ]:
print(fig)

In [ ]:
print(fig.generate_standalone())

This final figure uses the exact same ingredients as the earlier steps: declared
functions, composable expressions, sampled plots, and loops. The result is a
stylized tokamak X-point drawing with a separatrix-like structure, built
entirely from plain TikZ plots.

## A flux-surface-aligned curvilinear mesh

The same API can also draw a mesh aligned to an X-point map.  The next example
adds the patch layout with native `filldraw` and `grid` calls, then uses
`parametric_grid(...)` to draw both coordinate families of the mapped mesh.

In [ ]:
def add_eight_patch_background(fig, radius=6, color="Azure4"):
    patches = [
        [(0, 0), (radius, 0), (radius, radius), (0, radius)],
        [(0, 0), (-radius, 0), (-radius, radius), (0, radius)],
        [(0, 0), (radius, 0), (radius, -radius), (0, -radius)],
        [(0, 0), (-radius, 0), (-radius, -radius), (0, -radius)],
        [(radius, 0), (2 * radius, 0), (2 * radius, radius), (radius, radius)],
        [(-2 * radius, 0), (-radius, 0), (-radius, radius), (-2 * radius, radius)],
        [(0, radius), (radius, radius), (radius, 2 * radius), (0, 2 * radius)],
        [(0, -2 * radius), (radius, -2 * radius), (radius, -radius), (0, -radius)],
    ]
    for patch in patches:
        fig.filldraw(
            patch,
            cycle=True,
            fill=color,
            fill_opacity=0.06,
            draw=color,
            draw_opacity=0.35,
            line_width="0.4pt",
        )


fig = TikzFigure(figsize=(8, 8))
add_eight_patch_background(fig)
fig.grid((-6, -6), (6, 6), step="1", color=colors.Azure4, opacity=0.25)

The square-root map and the smooth blending function are declared once as PGF
functions.  `Var` keeps the expressions symbolic until TikZ samples the plots.

In [ ]:
arg_eta = Var("argEta")
arg_xi = Var("argXi")
x_mesh = Var("sampleParam")
x_mesh_two = Var("sampleParamTwo")
u_max = 6
v_max = 6

map_x = fig.function(
    "map_x", ("x", "y"), sqrt(max(0, sqrt(Var("x") ** 2 + Var("y") ** 2) - Var("x")))
)
map_y = fig.function(
    "map_y", ("x", "y"), sqrt(max(0, sqrt(Var("x") ** 2 + Var("y") ** 2) + Var("x")))
)
mapped_x = lambda a, b: sqrt(max(0, sqrt(a**2 + b**2) - a))
mapped_y = lambda a, b: sqrt(max(0, sqrt(a**2 + b**2) + a))
fig.function("smoothstep", "x", r"6*\x^5 - 15*\x^4 + 10*\x^3")
fig.function(
    "blend",
    "x",
    r"ifthenelse(\x < 0, 0, ifthenelse(\x > 1, 1, smoothstep(\x)))",
)
fig.function("q", "x", r"blend(2*(\x - 0.4))")

`parametric_grid` emits one loop for fixed parameter values and one for the
other parameter family.  The outer sign loops reuse the first-quadrant map in
all four sectors; the second call draws its coordinate-swapped counterpart.

In [ ]:
u_values = [i / 6 for i in range(7)]
v_values = [i / 6 for i in range(7)]
arg_eta_two = Var("argEtaTwo")

fig.parametric_grid(
    mapped_x(arg_eta, x_mesh),
    mapped_y(arg_eta, x_mesh),
    u_variable="argEta",
    v_variable="sampleParam",
    u_values=u_values,
    v_values=v_values,
    samples=18,
    smooth=True,
    options=[colors.Azure4],
    line_width="0.7pt",
)
fig.parametric_grid(
    mapped_y(arg_eta_two, x_mesh_two),
    mapped_x(arg_eta_two, x_mesh_two),
    u_variable="argEtaTwo",
    v_variable="sampleParamTwo",
    u_values=u_values,
    v_values=v_values,
    samples=18,
    smooth=True,
    options=[colors.Azure4],
    line_width="0.7pt",
)

fig.node((-2, 0.5), content=r"$\times$", text=colors.black)
fig.node((-2, 0.0), content=r"$\bullet$", text=colors.Orange1)
fig.show()

In [ ]:
print(fig)

In [ ]:
print(fig.generate_standalone())